## The goal
Evaluate the trained checkpoint on held-out test interactions using **Hit Rate @ 10** and **NDCG @ 10**.

### Eval protocol (leave-one-out)
For each held-out test interaction `(user, positive_book)` where `rating >= 4`:
1. Compute the user's embedding from precomputed `train_user_features` (built only from train interactions — no leakage).
2. Score every book in the catalog: `score = user_emb · item_emb`.
3. Mask out books the user already interacted with in the **train split** (we already know about those — they aren't recommendations).
4. Take the top-10. Is `positive_book` in there? That's HR@10. Where did it rank? That's NDCG@10.
5. Average across many test interactions.

### Reference points
- **HR@10 = 0.0001** ≈ random (top-10 out of 1.78M items).
- **HR@10 > 0.10** is decent for two-tower recommenders at this scale.
- **HR@10 > 0.30** is strong — getting close to production-quality.
- **NDCG@10** is always ≤ HR@10 (rank-weighted), typically ~50-70% of HR@10.

In [1]:
import json
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch

from mybookrec import DATA_DIR
from mybookrec.model.towers import TwoTowerModel
from mybookrec.eval.metrics import hit_rate_at_k, ndcg_at_k

K = 10                 # top-K cutoff
N_TEST_PAIRS = 5000    # how many (user, positive) pairs to evaluate — sample for speed
USER_BATCH = 64        # how many users to score at once (B × 1.78M scores ~ 450 MB at B=64)
CHECKPOINT_PATH = DATA_DIR.parent / 'checkpoints' / 'two_tower_mac.pt'

if torch.backends.mps.is_available():
    device = 'mps'
elif torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'
print(f'Device: {device}')
print(f'Checkpoint: {CHECKPOINT_PATH}  exists={CHECKPOINT_PATH.exists()}')

Device: mps
Checkpoint: /Users/zain/MyBookRec/checkpoints/two_tower_mac.pt  exists=True


In [2]:
# Load and reconstruct the model from the saved config.
ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
config = ckpt['config']
print(f"Checkpoint config: {config}")
print(f"Trained on {ckpt.get('n_batches_trained', '?'):,} batches")
print(f"Saved val loss: {ckpt.get('val_loss')}")

model = TwoTowerModel(
    user_input_dim=config['user_input_dim'],
    item_input_dim=config['item_input_dim'],
    hidden_dims=tuple(config['hidden_dims']),
    dropout=config['dropout'],
).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Learned temperature: {model.log_temperature.exp().item():.2f}")

Checkpoint config: {'hidden_dims': (512, 256, 128), 'dropout': 0.1, 'user_input_dim': 779, 'item_input_dim': 395, 'batch_size': 1024, 'lr': 0.001, 'n_negatives': 4}
Trained on 55,914 batches
Saved val loss: 0.2649044059962034
Learned temperature: 28.05


In [3]:
# Load feature matrices (same as training).
transformed = DATA_DIR / 'transformed'
user_features_np = np.load(transformed / 'train_user_features.npy')
book_embeddings_np = np.load(transformed / 'book_embeddings.npy')
genre_matrix_np = np.load(transformed / 'genre_matrix.npy')
pages_vec_np = np.load(transformed / 'num_pages_normalized.npy')

user_features = torch.from_numpy(user_features_np).to(device).float()
del user_features_np
_emb = torch.from_numpy(book_embeddings_np).to(device).float()
_genre = torch.from_numpy(genre_matrix_np).to(device).float()
_pages = torch.from_numpy(pages_vec_np).to(device).float().reshape(-1, 1)
del book_embeddings_np, genre_matrix_np, pages_vec_np
item_features = torch.cat([_emb, _genre, _pages], dim=1).contiguous()
del _emb, _genre, _pages

n_users, _ = user_features.shape
n_books, _ = item_features.shape
print(f'user_features: {tuple(user_features.shape)}  item_features: {tuple(item_features.shape)}')

user_features: (783032, 779)  item_features: (1782579, 395)


In [4]:
# Precompute embeddings for every user and every book — these are static after training.
# All scoring during eval is then just a dot product against these.
t0 = time.time()
with torch.no_grad():
    item_embs = []
    for i in range(0, n_books, 8192):
        item_embs.append(model.encode_item(item_features[i:i+8192]))
    all_item_embs = torch.cat(item_embs, dim=0)
    del item_embs

    user_embs = []
    for i in range(0, n_users, 8192):
        user_embs.append(model.encode_user(user_features[i:i+8192]))
    all_user_embs = torch.cat(user_embs, dim=0)
    del user_embs

print(f'all_item_embs: {tuple(all_item_embs.shape)}')
print(f'all_user_embs: {tuple(all_user_embs.shape)}')
print(f'embeddings computed in {time.time()-t0:.1f}s')

all_item_embs: (1782579, 128)
all_user_embs: (783032, 128)
embeddings computed in 1.9s


In [5]:
# Build the train-split exclusion sets (so we mask books the user already interacted with).
# Same join pattern as TrainingPairsDataset but built once here, used only at scoring time.
with open(transformed / 'user_id_to_index.json') as f:
    user_id_to_index = json.load(f)
with open(transformed / 'book_id_to_index.json') as f:
    book_id_to_index = json.load(f)

user_map = pl.DataFrame({
    'user_id': list(user_id_to_index.keys()),
    'user_idx': list(user_id_to_index.values()),
}, schema={'user_id': pl.String, 'user_idx': pl.Int64})
book_map = pl.DataFrame({
    'book_id': list(book_id_to_index.keys()),
    'book_idx': list(book_id_to_index.values()),
}, schema={'book_id': pl.String, 'book_idx': pl.Int64})

def load_split(name):
    return (
        pl.scan_parquet(transformed / 'training_interactions.parquet')
        .filter(pl.col('data_split') == name)
        .select('user_id', 'book_id', 'rating')
        .with_columns(pl.col('book_id').cast(pl.String))
        .join(user_map.lazy(), on='user_id', how='left')
        .join(book_map.lazy(), on='book_id', how='left')
        .filter(pl.col('user_idx').is_not_null() & pl.col('book_idx').is_not_null())
        .collect()
    )

t0 = time.time()
train_df = load_split('train')
print(f'train: {len(train_df):,} rows ({time.time()-t0:.1f}s)')

t0 = time.time()
train_exclude_groups = train_df.group_by('user_idx').agg(pl.col('book_idx').alias('rated_books'))
train_exclude = {
    row['user_idx']: np.array(row['rated_books'], dtype=np.int64)
    for row in train_exclude_groups.to_dicts()
}
print(f'exclude dict for {len(train_exclude):,} users ({time.time()-t0:.1f}s)')
del train_df, train_exclude_groups

train: 72,481,122 rows (10.0s)


exclude dict for 783,032 users (7.3s)


In [6]:
# Build the held-out test positives — these are the (user, book) pairs the model has never seen.
test_df = load_split('test')
test_pos = test_df.filter(pl.col('rating') >= 4)
print(f'Test positives: {len(test_pos):,}')

# Sample N pairs uniformly at random for evaluation.
rng = np.random.default_rng(0)
sample_idx = rng.choice(len(test_pos), size=min(N_TEST_PAIRS, len(test_pos)), replace=False)
sample = test_pos[sample_idx]
test_user_idxs = sample['user_idx'].to_numpy()
test_book_idxs = sample['book_idx'].to_numpy()
print(f'Sampled {len(sample):,} test pairs for evaluation')

Test positives: 14,093,590
Sampled 5,000 test pairs for evaluation


In [7]:
# Score each test user against the whole catalog, mask their train-rated books, compute HR/NDCG.
hits_list = []
ndcgs_list = []
t0 = time.time()
with torch.no_grad():
    for start in range(0, len(sample), USER_BATCH):
        end = min(start + USER_BATCH, len(sample))
        u_batch = test_user_idxs[start:end]
        b_batch = test_book_idxs[start:end]

        user_embs_b = all_user_embs[u_batch]                    # (B, 128)
        scores = user_embs_b @ all_item_embs.T                   # (B, 1.78M)

        # Mask train-rated books per user. CPU->GPU index assignment is fine for the small per-user sets.
        for i, u in enumerate(u_batch):
            exclude = train_exclude.get(int(u))
            if exclude is not None and len(exclude) > 0:
                scores[i, exclude] = float('-inf')

        targets_t = torch.from_numpy(b_batch).to(device)
        hits_list.append(hit_rate_at_k(scores, targets_t, k=K).cpu())
        ndcgs_list.append(ndcg_at_k(scores, targets_t, k=K).cpu())

        if (start // USER_BATCH) % 10 == 0:
            elapsed = time.time() - t0
            done = end
            print(f'  {done:5d} / {len(sample)}  ({elapsed:.0f}s, {done/elapsed:.0f} users/s)')

hits = torch.cat(hits_list)
ndcgs = torch.cat(ndcgs_list)
print(f'\nEvaluated {len(hits)} pairs in {time.time()-t0:.0f}s')

/tmp/claude-501/ipykernel_29126/1391477658.py:11: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  user_embs_b = all_user_embs[u_batch]                    # (B, 128)


     64 / 5000  (5s, 12 users/s)


    704 / 5000  (10s, 74 users/s)


   1344 / 5000  (13s, 100 users/s)


   1984 / 5000  (17s, 115 users/s)


   2624 / 5000  (21s, 124 users/s)


   3264 / 5000  (25s, 130 users/s)


   3904 / 5000  (29s, 133 users/s)


   4544 / 5000  (33s, 137 users/s)



Evaluated 5000 pairs in 36s


In [8]:
hr_mean = hits.float().mean().item()
ndcg_mean = ndcgs.mean().item()
ndcg_among_hits = ndcgs[hits].mean().item() if hits.any() else 0.0

print(f'Hit Rate @ {K}:        {hr_mean:.4f}   ({hr_mean*100:.2f}% of test books found in top {K})')
print(f'NDCG @ {K}:            {ndcg_mean:.4f}')
print(f'NDCG (among hits):    {ndcg_among_hits:.4f}  (average rank quality when we do hit)')
print(f'Random baseline HR@{K}: {K / n_books:.6f}  ({K} of {n_books:,} items)')

Hit Rate @ 10:        0.0100   (1.00% of test books found in top 10)
NDCG @ 10:            0.0043
NDCG (among hits):    0.4301  (average rank quality when we do hit)
Random baseline HR@10: 0.000006  (10 of 1,782,579 items)
